# فصل ۱۲: آشنایی با پایگاه داده‌های برداری (Vector Databases)
## بخش ۴: راه‌اندازی پایگاه‌داده برداری پایدار (Persistent ChromaDB Setup)

بسیاری از آموزش‌ها پایگاه‌های داده برداری را صرفاً در حالت **حافظه موقت (In-Memory)** نشان می‌دهند؛ در این حالت با بسته شدن اسکریپت یا خاموش شدن سیستم، تمامی بردارهای گران‌قیمت تولیدشده پاک می‌شوند!

در این بخش یاد می‌گیریم که چگونه یک پایگاه داده برداری پایدار با **ChromaDB** و **LangChain** بسازیم که اطلاعات را روی **دیسک سخت (Disk)** ذخیره کند تا در جلسات بعدی بدون نیاز به بردارسازی مجدد، مستقیماً لود شوند.

### ۱. آماده‌سازی مترجم: مدل تعبیه متن (Embedding Function)

پایگاه داده برداری متن طبیعی انسان را درک نمی‌کند؛ تنها بردارهای عددی را می‌شناسد. بنابراین پیش از هر کاری به یک «مترجم» یا تابع تعبیه نیاز داریم.

ما از مدل چندزبانه و محلی **HuggingFace** (`paraphrase-multilingual-MiniLM-L12-v2`) استفاده می‌کنیم که روی سیستم شما کش شده، از زبان فارسی پشتیبانی کامل دارد و **کاملاً پایدار و بدون وابستگی به اینترنت، فیلترشکن یا محدودیت تحریمی API** اجرا می‌شود.

> *(کد معادل ابری OpenAI نیز در صورت دسترسی به کلید API در کامنت قرار داده شده است)*

In [ ]:
import os
from dotenv import load_dotenv

# حذف پروکسی‌های غیرفعال محیطی
for var in ["http_proxy", "https_proxy", "HTTP_PROXY", "HTTPS_PROXY"]:
    os.environ.pop(var, None)

# فعال‌سازی حالت آفلاین برای لود آنی مدل کش‌شده
os.environ["HF_HUB_OFFLINE"] = "1"
load_dotenv()

# راه‌اندازی مدل تعبیه متن محلی چندزبانه
from langchain_huggingface import HuggingFaceEmbeddings

embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
print("✅ Embedding Function Ready!")

# ------------------------------------------------------------
# کد معادل برای مدل ابری OpenAI (در صورت دسترسی):
# from langchain_openai import OpenAIEmbeddings
# embedding_function = OpenAIEmbeddings(
#     base_url="https://api.tapsage.com/openai/v1",
#     api_key=os.getenv("OPENAI_API_KEY", "YOUR_API_KEY"),
#     model="text-embedding-3-small"
# )
# ------------------------------------------------------------

d:\Project\LLM\llm-apps-course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2733.62it/s]

### ۲. ساخت پایگاه داده برداری روی دیسک (Persistent ChromaDB)

برای ساخت پایگاه برداری پایدار، پارامتر **`persist_directory`** را مشخص می‌کنیم:

- **`collection_name`**: همانند نام جدول (Table) در SQL؛ می‌توانید چندین کالکشن مختلف (مثلاً مقالات، پرسش‌های متداول، محصولات) داشته باشید.
- **`embedding_function`**: تابعی که متن‌های ورودی را به بردار تبدیل می‌کند.
- **`persist_directory`**: مسیری روی دیسک که فایل دیتابیس `chroma.sqlite3` و فایل‌های باینری ایندکس HNSW در آن ذخیره می‌شوند.

> **نکته کلیدی:** اگر این پوشه وجود نداشته باشد، Chroma آن را می‌سازد؛ اگر از قبل وجود داشته باشد، داده‌های قبلی را به طور خودکار بارگذاری (Load) می‌کند.

In [ ]:
from langchain_chroma import Chroma

# مسیر ذخیره‌سازی پایگاه داده روی دیسک
persist_directory = "./chroma_db"

# ساخت (یا بارگذاری) دیتابیس پایدار با تنظیم فضای کسینوسی
vector_db = Chroma(
    collection_name="space_and_science_collection",
    embedding_function=embedding_function,
    persist_directory=persist_directory,
    collection_metadata={"hnsw:space": "cosine"}
)

print(f"✅ Database initialized at {persist_directory}")

✅ Database initialized at ./chroma_db


### ۳. تزریق اولین داده‌ها (Adding Documents, Metadata, and IDs)

اکنون چند فکت علمی-فضایی را به عنوان اسناد خام به همراه متادیتا و شناسه‌های یکتا (IDs) وارد دیتابیس می‌کنیم.

وقتی دستور `add_texts` را فراخوانی می‌کنیم، LangChain و Chroma مراحل زیر را به طور خودکار و در پشت صحنه انجام می‌دهند:
1. متن‌ها را برای `embedding_function` ارسال می‌کنند.
2. بردارهای عددی متناظر را دریافت می‌کنند.
3. متن خام و متادیتاها را در پایگاه رابطه‌ای `chroma.sqlite3` ذخیره می‌کنند.
4. بردارها را به ساختار گراف **HNSW** اضافه کرده و اتصالات چندلایه‌ای را به‌روزرسانی می‌کنند.

In [ ]:
# متون خام برای تست
documents = [
    "مریخ سیاره سرخ است و چهارمین سیاره منظومه شمسی می‌باشد.",
    "ماه تنها قمر طبیعی زمین است و انسان در سال ۱۹۶۹ روی آن پا گذاشت.",
    "خورشید یک ستاره است که ۹۹ درصد جرم منظومه شمسی را تشکیل می‌دهد.",
    "اقیانوس‌های زمین هنوز تا حد زیادی ناشناخته باقی مانده‌اند.",
    "هوش مصنوعی علمی است که سعی دارد هوش انسان را شبیه‌سازی کند."
]

# متادیتا (اطلاعات جانبی برای فیلترینگ یا استناد)
metadatas = [
    {"source": "space_book", "page": 10},
    {"source": "history_book", "page": 5},
    {"source": "space_book", "page": 12},
    {"source": "earth_book", "page": 1},
    {"source": "tech_book", "page": 100},
]

# شناسه‌های صریح (IDs) - اگر تعیین نکنیم کروما UUID تصادفی می‌سازد
ids = ["doc1", "doc2", "doc3", "doc4", "doc5"]

print("در حال افزودن اسناد به پایگاه داده...")
vector_db.add_texts(
    texts=documents,
    metadatas=metadatas,
    ids=ids
)
print("✅ Documents added successfully!")

در حال افزودن اسناد به پایگاه داده...
✅ Documents added successfully!


### ۴. آزمون جستجوی معنایی (Semantic Search)

اکنون سوالی مطرح می‌کنیم که کلمه کلیدی «مریخ» مستقیماً در آن وجود ندارد:
> **سوال:** «سیاره همسایه زمین که رنگش قرمز است»

بیایید ببینیم دیتابیس برداری چگونه مفهوم «سیاره سرخ» را استخراج می‌کند:

In [ ]:
query = "سیاره همسایه زمین که رنگش قرمز است"

# k=2 یعنی دو سند با بیشترین شباهت معنایی را بازگردان
results = vector_db.similarity_search(query, k=2)

print("--- Search Results ---\n")
for idx, res in enumerate(results, 1):
    print(f"نتیجه {idx}:")
    print(f"* متن سند: {res.page_content}")
    print(f"  متادیتا: {res.metadata}")
    print("-" * 40)

--- Search Results ---

نتیجه 1:
* متن سند: اقیانوس‌های زمین هنوز تا حد زیادی ناشناخته باقی مانده‌اند.
  متادیتا: {'source': 'earth_book', 'page': 1}
----------------------------------------
نتیجه 2:
* متن سند: مریخ سیاره سرخ است و چهارمین سیاره منظومه شمسی می‌باشد.
  متادیتا: {'source': 'space_book', 'page': 10}
----------------------------------------


### ۵. کالبدشکافی ساختار پوشه `chroma_db/` روی دیسک

اگر ساختار فایل‌های پوشه `./chroma_db` را بررسی کنیم، متوجه معماری دو لایه‌ای آن می‌شویم:

In [ ]:
import os

print(f"محتویات پوشه ذخیره‌سازی دیتابیس ({persist_directory}):\n")
if os.path.exists(persist_directory):
    for root, dirs, files in os.walk(persist_directory):
        level = root.replace(persist_directory, '').count(os.sep)
        indent = ' ' * 4 * level
        print(f"{indent}📂 {os.path.basename(root)}/")
        sub_indent = ' ' * 4 * (level + 1)
        for f in files:
            print(f"{sub_indent}📄 {f}")
else:
    print("پوشه هنوز ایجاد نشده است.")

محتویات پوشه ذخیره‌سازی دیتابیس (./chroma_db):

📂 chroma_db/
    📄 chroma.sqlite3
    📂 b333cc2d-1b60-4e1e-b121-e974eb33454d/
        📄 data_level0.bin
        📄 header.bin
        📄 length.bin
        📄 link_lists.bin


> **توضیح فایل‌ها:**
> 1. **`chroma.sqlite3`:** دیتابیس رابطه‌ای که اطلاعات کالکشن‌ها، متن‌های خام (`documents`) و فراداده‌ها (`metadata`) را در خود ذخیره می‌کند.
> 2. **پوشه‌های دارای شناسه (UUID):** محل ذخیره‌سازی داده‌های باینری و گراف **HNSW** (شاخص‌های برداری).
>
> ⚠️ **نکته بسیار مهم گیت (Git):**
> دیتابیس‌های برداری به دلیل حجم بالا و باینری بودن نباید در مخزن گیت کامیت شوند. حتماً پوشه `chroma_db/` و فایل‌های `*.sqlite3` را در فایل **`.gitignore`** پروژه قرار دهید.

### ۶. آزمون ماندگاری و بارگذاری مجدد (Persistence Verification)

برای اثبات اینکه داده‌ها روی دیسک ذخیره شده‌اند و نیازی به فراخوانی مجدد `add_texts` نیست، یک شیء جدید از دیتابیس می‌سازیم و بلافاصله جستجو را اجرا می‌کنیم:

In [ ]:
# بارگذاری مجدد دیتابیس از همان مسیر بدون اضافه کردن داده جدید
reloaded_db = Chroma(
    collection_name="space_and_science_collection",
    embedding_function=embedding_function,
    persist_directory=persist_directory
)

# تست بازیابی سند هوش مصنوعی
test_query = "ماشین‌هایی که قدرت تفکر و یادگیری پیدا می‌کنند"
reloaded_results = reloaded_db.similarity_search_with_score(test_query, k=1)

doc, score = reloaded_results[0]
print("✅ بازیابی موفقیت‌آمیز از روی دیسک:")
print(f"متن بازیابی‌شده: {doc.page_content}")
print(f"متادیتا:         {doc.metadata}")
print(f"فاصله کسینوسی:   {score:.4f}")

✅ بازیابی موفقیت‌آمیز از روی دیسک:
متن بازیابی‌شده: هوش مصنوعی علمی است که سعی دارد هوش انسان را شبیه‌سازی کند.
متادیتا:         {'page': 100, 'source': 'tech_book'}
فاصله کسینوسی:   0.4642


### ⚠️ چالش مهم و پیش‌درآمد درس بعدی (مدیریت چرخه داده‌ها - CRUD)

در کد بالا اگر سلول افزودن اسناد (`add_texts`) را چندین بار پشت سر هم بدون تغییر شناسه اجرا کنید، ممکن است داده‌ها تکرار شوند یا خطای شناسه بگیرید.

در سیستم‌های تجاری واقعی به مدیریت کامل داده‌ها نیاز داریم:
- چگونه داده‌های موجود را مشاهده و شمارش کنیم؟ (`get`)
- چگونه یک سند تغییر یافته را اصلاح کنیم؟ (`update`)
- چگونه اسناد منقضی یا نامعتبر را پاک کنیم؟ (`delete`)

در درس‌نامه آینده به طور جامع عملیات **CRUD** را در پایگاه‌های داده برداری بررسی خواهیم کرد.